# 02 - Candidate Variable Acquisition
Main goal: Use our raw data to calculate our candidate variables

In [68]:
import pandas as pd
import numpy as np
import os
from scipy.stats import mstats
os.chdir('/Users/thomas/Documents/GitHub/sv_project')

In [69]:
#Loading SPX return data
spx = pd.read_csv('data/raw/01_spx_returns.csv')
spx = spx[['Date', 'Returns']]
spx

,Date,Returns
0,1996-01-03,0.000951
1,1996-01-04,-0.005826
2,1996-01-05,-0.001603
3,1996-01-08,0.002838
4,1996-01-09,-0.014568
...,...,...
7651,2026-06-01,0.002625
7652,2026-06-02,0.001292
7653,2026-06-03,-0.007372
7654,2026-06-04,0.004055


In [70]:
#Return-based volatility and downside-risk variables

#Annualized square return
spx['r2_ann'] = (spx['Returns']**2)*252
# spx['r2_ann_winsorized'] = mstats.winsorize(spx['r2_ann'], limits=[0.0, 0.005])

#Rolling RV
for window in [5, 21, 63, 126, 252]:
    spx[f'RV_{window}'] = spx['r2_ann'].rolling(window).mean()

#EWMA
for lam in [0.94, 0.97, 0.985]:
    spx[f'EWMA_lam_{lam}'] = spx['r2_ann'].ewm(alpha = 1-lam, adjust = False).mean()

#Downside RV
spx['RV_down'] = spx['r2_ann'].where(spx['Returns'] < 0, 0)
for window in [5, 21, 63]: 
    spx[f'RV_down_{window}'] = spx['RV_down'].rolling(window).mean()

#Return-shape controls
spx['abs_ret_norm'] = spx['Returns'].abs() * np.sqrt(252)
spx['neg_ret_dummy'] = np.where(spx['Returns'] < 0, 1, 0)
spx['neg_ret_mag'] = spx['Returns'].where(spx['Returns'] < 0, 0).abs()
spx.to_csv('data/processed/SPX_transformed.csv', index=False)
spx

,Date,Returns,r2_ann,RV_5,RV_21,RV_63,RV_126,RV_252,EWMA_lam_0.94,EWMA_lam_0.97,EWMA_lam_0.985,RV_down,RV_down_5,RV_down_21,RV_down_63,abs_ret_norm,neg_ret_dummy,neg_ret_mag
0,1996-01-03,0.000951,0.000228,NaN,NaN,NaN,NaN,NaN,0.000228,0.000228,0.000228,0.000000,NaN,NaN,NaN,0.015089,0,0.000000
1,1996-01-04,-0.005826,0.008554,NaN,NaN,NaN,NaN,NaN,0.000727,0.000477,0.000353,0.008554,NaN,NaN,NaN,0.092490,1,0.005826
2,1996-01-05,-0.001603,0.000647,NaN,NaN,NaN,NaN,NaN,0.000722,0.000483,0.000357,0.000647,NaN,NaN,NaN,0.025442,1,0.001603
3,1996-01-08,0.002838,0.002029,NaN,NaN,NaN,NaN,NaN,0.000801,0.000529,0.000382,0.000000,NaN,NaN,NaN,0.045046,0,0.000000
4,1996-01-09,-0.014568,0.053484,0.012989,NaN,NaN,NaN,NaN,0.003962,0.002118,0.001179,0.053484,0.012537,NaN,NaN,0.231267,1,0.014568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7651,2026-06-01,0.002625,0.001737,0.004136,0.010819,0.021278,0.016274,0.014290,0.012726,0.016233,0.017873,0.000000,0.000000,0.002778,0.007701,0.041675,0,0.000000
7652,2026-06-02,0.001292,0.000421,0.002339,0.010736,0.020928,0.016220,0.014292,0.011987,0.015758,0.017611,0.000000,0.000000,0.002778,0.007344,0.020511,0,0.000000
7653,2026-06-03,-0.007372,0.013695,0.005077,0.011190,0.020904,0.016272,0.014329,0.012090,0.015697,0.017552,0.013695,0.002739,0.003232,0.007561,0.117028,1,0.007372
7654,2026-06-04,0.004055,0.004144,0.004237,0.010596,0.020843,0.016293,0.014312,0.011613,0.015350,0.017351,0.000000,0.002739,0.003232,0.007434,0.064371,0,0.000000


In [71]:
#Market liquidity and trading frictions
df = pd.read_csv('data/raw/01_spx_returns.csv', parse_dates=['Date'])
df = df[['Date', 'Close', 'Volume', 'Returns']]

#SPX dollar volume
df['dollar_vol'] = df['Close'] * df['Volume']

#Rolling volume growth using the past 21 days
df['vol_growth'] = df['dollar_vol'].pct_change()
df['rolling_vol_growth'] = df['vol_growth'].rolling(21).mean()

#Amihud illiquidity and rolling 21 day amihud
df['amihud_illiquidity'] = df['Returns'].abs()/df['dollar_vol']
df['rolling_amihud'] = df['amihud_illiquidity'].rolling(21).mean()

df.to_csv('data/processed/liquidity.csv', index=False)
df

,Date,Close,Volume,Returns,dollar_vol,vol_growth,rolling_vol_growth,amihud_illiquidity,rolling_amihud
0,1996-01-03,621.320007,468950000,0.000951,2.913680e+11,NaN,NaN,3.262325e-15,NaN
1,1996-01-04,617.700012,512580000,-0.005826,3.166207e+11,0.086669,NaN,1.840151e-14,NaN
2,1996-01-05,616.710022,437110000,-0.001603,2.695701e+11,-0.148602,NaN,5.945406e-15,NaN
3,1996-01-08,618.460022,130360000,0.002838,8.062245e+10,-0.700922,NaN,3.519663e-14,NaN
4,1996-01-09,609.450012,417400000,-0.014568,2.543844e+11,2.155256,NaN,5.726946e-14,NaN
...,...,...,...,...,...,...,...,...,...
7651,2026-06-01,7599.959961,6583960000,0.002625,5.003783e+13,-0.159964,0.017065,5.246622e-17,1.281703e-16
7652,2026-06-02,7609.779785,5904410000,0.001292,4.493126e+13,-0.102054,0.019378,2.875702e-17,1.255610e-16
7653,2026-06-03,7553.680176,5753860000,-0.007372,4.346282e+13,-0.032682,0.014414,1.696172e-16,1.284872e-16
7654,2026-06-04,7584.310059,5398910000,0.004055,4.094701e+13,-0.057884,0.010679,9.902950e-17,1.231143e-16


In [72]:
#Loading FRED data
fred_raw = pd.read_csv('data/raw/02_fred_raw.csv', parse_dates=['Date'])
spx = pd.read_csv('data/raw/01_spx_returns.csv', parse_dates=['Date'])
spx = spx[['Date']].copy()

#Linking FRED data to SPX trading days using merge_asof to avoid look ahead bias
fred = pd.merge_asof(
    spx.sort_values('Date'),
    fred_raw.sort_values('Date'),
    on='Date'
)

fred

,Date,BAA,AAA,HY_OAS,IG_OAS,TED,SOFR,DGS3MO,DGS2,DGS10,DTB3
0,1996-01-03,7.37,6.71,NaN,NaN,0.58,NaN,5.20,5.17,5.58,5.05
1,1996-01-04,7.44,6.78,NaN,NaN,0.57,NaN,5.19,5.17,5.65,5.04
2,1996-01-05,7.50,6.80,NaN,NaN,0.58,NaN,5.19,5.20,5.69,5.03
3,1996-01-08,7.49,6.80,NaN,NaN,0.57,NaN,5.18,5.20,5.68,5.03
4,1996-01-09,7.46,6.79,NaN,NaN,0.59,NaN,5.18,5.18,5.70,5.01
...,...,...,...,...,...,...,...,...,...,...,...
7651,2026-06-01,6.01,5.49,2.72,0.73,NaN,3.65,3.78,4.05,4.47,3.63
7652,2026-06-02,6.00,5.48,2.71,0.74,NaN,3.63,3.77,4.05,4.46,3.63
7653,2026-06-03,6.03,5.51,2.75,0.74,NaN,3.61,3.78,4.08,4.49,3.63
7654,2026-06-04,6.01,5.49,2.74,0.74,NaN,3.62,3.78,4.05,4.47,3.63


In [73]:
#Transforming FRED data

#Investment grade and high yield credit spread
fred['baa_aaa'] = fred['BAA'] - fred['AAA']

#Daily change in credit spreads and absolute change
fred['baa_aaa_change'] = fred['baa_aaa'].diff()
fred['baa_aaa_change_abs'] = fred['baa_aaa_change'].abs()
fred['HY_OAS_change'] = fred['HY_OAS'].diff()
fred['HY_OAS_change_abs'] = fred['HY_OAS_change'].abs()
fred['IG_OAS_change'] = fred['IG_OAS'].diff()
fred['IG_OAS_change_abs'] = fred['IG_OAS_change'].abs()

#SOFR spread
fred['SOFR_spread'] = fred['SOFR'] - fred['DTB3']

#Yield slope (10y-2y)
fred['yield_slope'] = fred['DGS10'] - fred['DGS2']

#Treasury yield curvature using butterfly spread
fred['yield_curv'] = 2 * fred['DGS2'] - fred['DGS3MO'] - fred['DGS10']

#Rolling RV over 5, 21, 63, 126, and 252 trading days of daily changes in 10y and 2y
fred['d_DGS2'] = fred['DGS2'].diff()
fred['d_DGS10'] = fred['DGS10'].diff()
for window in [5, 21, 63, 126, 252]:
    fred[f'RV_rate_2y_{window}']  = (fred['d_DGS2']**2).rolling(window, min_periods=1).mean()
    fred[f'RV_rate_10y_{window}'] = (fred['d_DGS10']**2).rolling(window, min_periods=1).mean()

fred.to_csv('data/processed/FRED_transformed.csv', index=False)
fred

,Date,BAA,AAA,HY_OAS,IG_OAS,TED,SOFR,DGS3MO,DGS2,DGS10,...,RV_rate_2y_5,RV_rate_10y_5,RV_rate_2y_21,RV_rate_10y_21,RV_rate_2y_63,RV_rate_10y_63,RV_rate_2y_126,RV_rate_10y_126,RV_rate_2y_252,RV_rate_10y_252
0,1996-01-03,7.37,6.71,NaN,NaN,0.58,NaN,5.20,5.17,5.58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1996-01-04,7.44,6.78,NaN,NaN,0.57,NaN,5.19,5.17,5.65,...,0.000000,0.00490,0.000000,0.004900,0.000000,0.004900,0.000000,0.004900,0.000000,0.004900
2,1996-01-05,7.50,6.80,NaN,NaN,0.58,NaN,5.19,5.20,5.69,...,0.000450,0.00325,0.000450,0.003250,0.000450,0.003250,0.000450,0.003250,0.000450,0.003250
3,1996-01-08,7.49,6.80,NaN,NaN,0.57,NaN,5.18,5.20,5.68,...,0.000300,0.00220,0.000300,0.002200,0.000300,0.002200,0.000300,0.002200,0.000300,0.002200
4,1996-01-09,7.46,6.79,NaN,NaN,0.59,NaN,5.18,5.18,5.70,...,0.000325,0.00175,0.000325,0.001750,0.000325,0.001750,0.000325,0.001750,0.000325,0.001750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7651,2026-06-01,6.01,5.49,2.72,0.73,NaN,3.65,3.78,4.05,4.47,...,0.003920,0.00106,0.002929,0.002357,0.002717,0.002178,0.001971,0.001702,0.002217,0.001721
7652,2026-06-02,6.00,5.48,2.71,0.74,NaN,3.63,3.77,4.05,4.46,...,0.001040,0.00036,0.002929,0.002357,0.002692,0.002178,0.001968,0.001700,0.002213,0.001720
7653,2026-06-03,6.03,5.51,2.75,0.74,NaN,3.61,3.78,4.08,4.49,...,0.001200,0.00046,0.002738,0.002229,0.002692,0.002178,0.001937,0.001668,0.002206,0.001714
7654,2026-06-04,6.01,5.49,2.74,0.74,NaN,3.62,3.78,4.05,4.47,...,0.001360,0.00036,0.002762,0.002229,0.002692,0.002159,0.001937,0.001671,0.002208,0.001715


In [74]:
#Loading yfinance data
yf_raw = pd.read_csv('data/raw/03_yf_raw.csv', parse_dates=['Date'])

#Linking yfinance data to SPX trading days using merge_asof to avoid look ahead bias
yf = pd.merge_asof(
    spx.sort_values('Date'),
    yf_raw.sort_values('Date'),
    on='Date'
)
yf

,Date,HYG,LQD,XLF,WTI,USD,GOLD
0,1996-01-03,NaN,NaN,NaN,NaN,85.110001,NaN
1,1996-01-04,NaN,NaN,NaN,NaN,85.220001,NaN
2,1996-01-05,NaN,NaN,NaN,NaN,85.059998,NaN
3,1996-01-08,NaN,NaN,NaN,NaN,85.040001,NaN
4,1996-01-09,NaN,NaN,NaN,NaN,85.070000,NaN
...,...,...,...,...,...,...,...
7651,2026-06-01,79.839996,108.930000,51.430000,92.160004,99.199997,4475.200195
7652,2026-06-02,79.900002,108.919998,51.459999,93.760002,99.220001,4489.100098
7653,2026-06-03,79.680000,108.620003,50.869999,96.019997,99.529999,4436.700195
7654,2026-06-04,79.830002,108.849998,52.189999,93.040001,99.410004,4475.799805


In [75]:
#Transforming yfinance data

#Getting returns for each series for RV calculation
for col in ['HYG', 'LQD', 'XLF', 'WTI', 'USD', 'GOLD']:
    yf[f'{col}_ret'] = yf[col].pct_change()

#Calculating RV over 5, 21, 63, 126, and 252 trading days
for col in ['HYG', 'LQD', 'XLF', 'WTI', 'USD', 'GOLD']:
    for window in [5, 21, 63, 126, 252]:
        yf[f'{col}_RV{window}'] = (yf[f'{col}_ret']**2 * 252).rolling(window).mean()

yf.to_csv('data/processed/yf_transformed.csv', index=False)
yf

,Date,HYG,LQD,XLF,WTI,USD,GOLD,HYG_ret,LQD_ret,XLF_ret,...,USD_RV5,USD_RV21,USD_RV63,USD_RV126,USD_RV252,GOLD_RV5,GOLD_RV21,GOLD_RV63,GOLD_RV126,GOLD_RV252
0,1996-01-03,NaN,NaN,NaN,NaN,85.110001,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1996-01-04,NaN,NaN,NaN,NaN,85.220001,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1996-01-05,NaN,NaN,NaN,NaN,85.059998,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1996-01-08,NaN,NaN,NaN,NaN,85.040001,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1996-01-09,NaN,NaN,NaN,NaN,85.070000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7651,2026-06-01,79.839996,108.930000,51.430000,92.160004,99.199997,4475.200195,-0.000764,-0.000156,-0.002908,...,0.000803,0.001744,0.003371,0.003101,0.003512,0.041804,0.038684,0.079578,0.104318,0.071848
7652,2026-06-02,79.900002,108.919998,51.459999,93.760002,99.220001,4489.100098,0.000752,-0.000092,0.000583,...,0.000691,0.001724,0.003185,0.003097,0.003512,0.041244,0.038669,0.074626,0.104012,0.071786
7653,2026-06-03,79.680000,108.620003,50.869999,96.019997,99.529999,4436.700195,-0.002753,-0.002754,-0.011465,...,0.001174,0.001757,0.003192,0.003116,0.003482,0.041147,0.033481,0.075146,0.104235,0.071305
7654,2026-06-04,79.830002,108.849998,52.189999,93.040001,99.410004,4475.799805,0.001883,0.002117,0.025948,...,0.001063,0.001774,0.003074,0.003119,0.003452,0.038225,0.033639,0.074997,0.104081,0.071346


In [76]:
#Loading EPU data
epu = pd.read_csv('data/raw/EPU_index.csv')

#Assigning proper date format
epu['Date'] = pd.to_datetime(epu[['year', 'month', 'day']])
epu = epu[['Date', 'daily_policy_index']].copy()
epu = epu[epu['Date'] >= '1996-01-01'].reset_index(drop=True)

#Aligning dates to SPX trading days
epu = pd.merge_asof(
    spx.sort_values('Date'),
    epu.sort_values('Date'),
    on='Date'
)
epu.columns = ['Date', 'EPU']
epu.to_csv('data/processed/EPU.csv', index=False)
epu

,Date,EPU
0,1996-01-03,135.69
1,1996-01-04,93.54
2,1996-01-05,36.13
3,1996-01-08,136.18
4,1996-01-09,144.06
...,...,...
7651,2026-06-01,242.92
7652,2026-06-02,221.12
7653,2026-06-03,468.63
7654,2026-06-04,379.17


In [77]:
#Loading GPR data
gpr = pd.read_excel('data/raw/GPR_index.xls')

#Assigning proper date format
gpr = gpr[['month', 'GPR']].copy()
gpr.columns = ['Date', 'GPR']
gpr['Date'] = pd.to_datetime(gpr['Date'])
gpr = gpr[gpr['Date'] >= '1996-01-01'].reset_index(drop=True)

#Aligning to SPX trading days
gpr = pd.merge_asof(
    spx.sort_values('Date'),
    gpr.sort_values('Date'),
    on='Date'
)
gpr.to_csv('data/processed/GPR.csv', index=False)
gpr

,Date,GPR
0,1996-01-03,75.634926
1,1996-01-04,75.634926
2,1996-01-05,75.634926
3,1996-01-08,75.634926
4,1996-01-09,75.634926
...,...,...
7651,2026-06-01,184.177048
7652,2026-06-02,184.177048
7653,2026-06-03,184.177048
7654,2026-06-04,184.177048


In [79]:
# Combining all the data and creating the final CSV file

# Columns to drop - not candidate X variables and not the anchor
cols_to_drop = ['Returns', 'r2_ann', 'BAA', 'AAA', 'DTB3', 'SOFR',
                'd_DGS2', 'd_DGS10', 'HYG', 'LQD', 'XLF', 'WTI', 'USD', 'GOLD',
                'vol_growth', 'HYG_ret', 'LQD_ret', 'XLF_ret', 
                'WTI_ret', 'USD_ret', 'GOLD_ret']

# Columns to NOT standardize
do_not_standardize = ['Date', 'neg_ret_dummy']

# Load SPX transformed data as base
base = pd.read_csv('data/processed/SPX_transformed.csv', parse_dates=['Date'])

# Merge liquidity
liq = df.drop(columns=['Close', 'Volume', 'Returns'])

# Merge everything together
merged = base.merge(liq, on='Date', how='left')
merged = merged.merge(fred, on='Date', how='left')
merged = merged.merge(yf, on='Date', how='left')
merged = merged.merge(epu, on='Date', how='left')
merged = merged.merge(gpr, on='Date', how='left')

# Save unstandarized version
merged.to_csv('data/processed/all_variables_raw.csv', index=False)

# Drop non-candidate columns before standardizing
merged_std = merged.drop(columns=cols_to_drop)

# All other columns get standardized except those in do_not_standardize
cols_to_standardize = [col for col in merged_std.columns if col not in do_not_standardize]

# Rolling standardization using expanding window
for col in cols_to_standardize:
    rolling_mean = merged_std[col].expanding().mean()
    rolling_std = merged_std[col].expanding().std()
    merged_std[col] = (merged_std[col] - rolling_mean) / rolling_std

merged_std.to_csv('data/processed/all_variables_standardized.csv', index=False)
print(merged_std.shape)
merged_std

(7656, 78)


,Date,RV_5,RV_21,RV_63,RV_126,RV_252,EWMA_lam_0.94,EWMA_lam_0.97,EWMA_lam_0.985,RV_down,...,USD_RV63,USD_RV126,USD_RV252,GOLD_RV5,GOLD_RV21,GOLD_RV63,GOLD_RV126,GOLD_RV252,EPU,GPR
0,1996-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1996-01-04,NaN,NaN,NaN,NaN,NaN,0.707107,0.707107,0.707107,0.707107,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.707107,NaN
2,1996-01-05,NaN,NaN,NaN,NaN,NaN,0.568971,0.594743,0.607194,-0.508065,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.047000,NaN
3,1996-01-08,NaN,NaN,NaN,NaN,NaN,0.687476,0.732098,0.753577,-0.550280,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.757250,NaN
4,1996-01-09,NaN,NaN,NaN,NaN,NaN,1.768332,1.767386,1.766886,1.766891,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.770321,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7651,2026-06-01,-0.380098,-0.369606,-0.276093,-0.449148,-0.633276,-0.386291,-0.387546,-0.433082,-0.200246,...,-0.668762,-0.826942,-0.841882,0.199938,0.185375,1.666737,2.931987,1.852558,1.281829,1.563459
7652,2026-06-02,-0.401287,-0.370777,-0.282441,-0.450304,-0.633170,-0.398374,-0.396710,-0.439253,-0.200232,...,-0.719077,-0.828007,-0.841855,0.188724,0.184939,1.489529,2.916546,1.848505,1.059195,1.563108
7653,2026-06-03,-0.368918,-0.364177,-0.282844,-0.449120,-0.632063,-0.396662,-0.397880,-0.440611,-0.049063,...,-0.717080,-0.822215,-0.852348,0.186778,0.033742,1.507638,2.922856,1.824043,3.581980,1.562756
7654,2026-06-04,-0.378811,-0.372746,-0.283948,-0.448624,-0.632483,-0.404455,-0.404568,-0.445343,-0.200225,...,-0.749140,-0.821402,-0.862536,0.128340,0.038340,1.501867,2.913694,1.825090,2.668263,1.562405
